# Week 7 — RFM Customer Segmentation for UrbanStyle
Team notebook: data cleaning → RFM scores → customer segments → charts for Marko.

In [1]:
import pandas as pd

df = pd.read_csv("../data/sales_clean.csv")
print("Initial shape:", df.shape)
df.head()


Initial shape: (8950, 13)


,sale_id,invoice_id,sale_date,customer_id,product_id,quantity,unit_price,total_price,channel,store_location,payment_method,first_name,email
0,1,INV-202301-00001,2023-01-10,2588,1274,2,234.79,469.58,pood,Tallinn,kaart,Hille,NaN
1,2,INV-202301-00002,2023-01-16,4338,1207,2,241.13,482.26,pood,Pärnu,järelmaks,Merle,merle.luik@mail.ee
2,3,INV-202301-00003,2023-01-05,4673,1264,1,258.46,221.19,pood,Pärnu,järelmaks,Liina,liina.saar@gmail.com
3,4,INV-202301-00004,2023-01-02,4677,1341,3,45.21,135.63,pood,Tartu,sularaha,Aili,aili.pihl@yahoo.com
4,5,INV-202301-00005,2023-01-13,2390,1284,1,99.57,99.57,pood,Tartu,kaart,Triin,triin.lill@telia.ee


In [2]:
dup_count = df.duplicated(subset=["invoice_id"]).sum()
print("Duplicate invoices:", dup_count)

df = df.drop_duplicates(subset=["invoice_id"], keep="first")
print("Shape after dedupe:", df.shape)


Duplicate invoices: 0
Shape after dedupe: (8950, 13)


In [3]:
print("NULLs per column:\n", df.isnull().sum())

df = df.dropna(subset=["customer_id", "sale_date", "total_price"])
print("Shape after NULL removal:", df.shape)


NULLs per column:
 sale_id              0
invoice_id           0
sale_date            0
customer_id          0
product_id           0
quantity             0
unit_price           0
total_price          0
channel              0
store_location    3054
payment_method       0
first_name           0
email              939
dtype: int64
Shape after NULL removal: (8950, 13)


In [4]:
df["sale_date"] = pd.to_datetime(df["sale_date"], errors="coerce")

bad_dates = df["sale_date"].isnull().sum()
print("Unparseable dates:", bad_dates)

df = df.dropna(subset=["sale_date"])
print("Shape after date parsing:", df.shape)
print("sale_date dtype:", df["sale_date"].dtype)


Unparseable dates: 0
Shape after date parsing: (8950, 13)
sale_date dtype: datetime64[us]


In [5]:
neg_count = (df["total_price"] <= 0).sum()
print("Non-positive total_price rows:", neg_count)

df = df[df["total_price"] > 0]
print("Shape after outlier removal:", df.shape)


Non-positive total_price rows: 0
Shape after outlier removal: (8950, 13)


In [6]:
final_rows = df.shape[0]
unique_customers = df["customer_id"].nunique()
date_min = df["sale_date"].min()
date_max = df["sale_date"].max()

print("\n=== CLEANING REPORT ===")
print(f"Final shape: {final_rows} rows")
print(f"Unique customers: {unique_customers}")
print(f"Date range: {date_min.date()} → {date_max.date()}")
print(f"Duplicates removed: {dup_count}")
print("Critical NULLs remaining:",
      df[["customer_id", "sale_date", "total_price"]].isnull().sum().to_dict())
print("Non-positive total_price remaining:",
      (df["total_price"] <= 0).sum())



=== CLEANING REPORT ===
Final shape: 8950 rows
Unique customers: 2540
Date range: 2023-01-01 → 2026-06-28
Duplicates removed: 0
Critical NULLs remaining: {'customer_id': 0, 'sale_date': 0, 'total_price': 0}
Non-positive total_price remaining: 0


## RFM Analysis

In [7]:
today = df['sale_date'].max() + pd.Timedelta(days=1)


In [8]:
recency = (
    df.groupby("customer_id")["sale_date"]
      .max()
      .reset_index()
      .rename(columns={"sale_date": "last_purchase_date"})
)

recency["Recency"] = (today - recency["last_purchase_date"]).dt.days
recency.head()


,customer_id,last_purchase_date,Recency
0,2001,2024-11-29,577
1,2004,2024-12-19,557
2,2005,2024-10-03,634
3,2006,2023-11-09,963
4,2007,2025-01-30,515


In [9]:
frequency = (
    df.groupby("customer_id")["sale_id"]
      .count()
      .reset_index()
      .rename(columns={"sale_id": "Frequency"})
)

frequency.head()


,customer_id,Frequency
0,2001,2
1,2004,2
2,2005,4
3,2006,1
4,2007,1


In [10]:
monetary = (
    df.groupby("customer_id")["total_price"]
      .sum()
      .reset_index()
      .rename(columns={"total_price": "Monetary"})
)

monetary.head()


,customer_id,Monetary
0,2001,203.92
1,2004,1198.56
2,2005,959.60
3,2006,327.06
4,2007,318.63


In [11]:
rfm = recency.merge(frequency, on="customer_id") \
             .merge(monetary, on="customer_id")

rfm.head()


,customer_id,last_purchase_date,Recency,Frequency,Monetary
0,2001,2024-11-29,577,2,203.92
1,2004,2024-12-19,557,2,1198.56
2,2005,2024-10-03,634,4,959.60
3,2006,2023-11-09,963,1,327.06
4,2007,2025-01-30,515,1,318.63


In [12]:
# Recency score (lower recency = better)
rfm["R_score_raw"] = pd.qcut(rfm["Recency"], 5, labels=False) + 1
rfm["R_score"] = 6 - rfm["R_score_raw"]

# Frequency score (rank first to avoid ties)
rfm["Frequency_rank"] = rfm["Frequency"].rank(method="first")
rfm["F_score"] = pd.qcut(rfm["Frequency_rank"], 5, labels=False) + 1

# Monetary score
rfm["M_score"] = pd.qcut(rfm["Monetary"], 5, labels=False) + 1


In [13]:
rfm["RFM_Score"] = rfm["R_score"] + rfm["F_score"] + rfm["M_score"]

def assign_segment(score):
    if score >= 13:
        return "VIP Champions"
    elif score >= 10:
        return "Loyal"
    elif score >= 7:
        return "Potential"
    elif score >= 4:
        return "At Risk"
    else:
        return "Lost"

rfm["Segment"] = rfm["RFM_Score"].apply(assign_segment)

rfm.head()


,customer_id,last_purchase_date,Recency,Frequency,Monetary,R_score_raw,R_score,Frequency_rank,F_score,M_score,RFM_Score,Segment
0,2001,2024-11-29,577,2,203.92,2,4,479.0,1,1,6,At Risk
1,2004,2024-12-19,557,2,1198.56,2,4,480.0,1,4,9,Potential
2,2005,2024-10-03,634,4,959.60,3,3,1618.0,4,3,10,Loyal
3,2006,2023-11-09,963,1,327.06,5,1,1.0,1,1,3,Lost
4,2007,2025-01-30,515,1,318.63,1,5,2.0,1,1,7,Potential


In [14]:
rfm["Segment"].value_counts()


Segment
Potential        759
Loyal            679
At Risk          529
VIP Champions    455
Lost             118
Name: count, dtype: int64

# ROLE D — Visualization (RFM Insights)


In [15]:
import plotly.express as px


In [16]:
seg_counts = rfm["Segment"].value_counts().reset_index()
seg_counts.columns = ["Segment", "Count"]

fig1 = px.bar(
    seg_counts,
    x="Segment",
    y="Count",
    title="Customer Count by Segment",
    labels={"Segment": "Segment", "Count": "Customer Count"}
)

fig1.show()


In [17]:
fig2 = px.scatter(
    rfm,
    x="Recency",
    y="Monetary",
    color="Segment",
    size="Frequency",
    title="Customer Segments (RFM Scatter Plot)",
    labels={
        "Recency": "Days Since Last Purchase",
        "Monetary": "Total Spend (EUR)",
        "Frequency": "Purchase Count"
    }
)

fig2.show()


In [18]:
top_vips = (
    rfm[rfm["Segment"] == "VIP Champions"]
    .nlargest(10, "Monetary")
)
top_vips["customer_id"] = top_vips["customer_id"].astype(str)

fig3 = px.bar(
    top_vips,
    x="customer_id",
    y="Monetary",
    title="Top 10 VIP Customers by Total Spend",
    labels={"customer_id": "Customer ID", "Monetary": "Total Spend (EUR)"}
)

fig3.show()


In [19]:
summary = rfm.groupby("Segment").agg(
    customers=("customer_id", "count"),
    revenue=("Monetary", "sum")
)
summary["% of customers"] = (summary["customers"] / summary["customers"].sum() * 100).round(1)
summary["% of revenue"] = (summary["revenue"] / summary["revenue"].sum() * 100).round(1)
summary.sort_values("revenue", ascending=False)

,customers,revenue,% of customers,% of revenue
Segment,,,,
VIP Champions,455,1146295.15,17.9,42.8
Loyal,679,796357.18,26.7,29.7
Potential,759,521792.88,29.9,19.5
At Risk,529,192170.22,20.8,7.2
Lost,118,20235.11,4.6,0.8


## Summary for Marko

**Key finding:** VIP Champions are 18% of customers but bring 43% of revenue. Together with Loyal customers, 45% of customers generate 73% of revenue.

**Urgent:** 529 customers (21%) are At Risk. They have spent €192k so far but have not bought recently.

**Recommendations:**
1. **VIP programme** – early access and exclusive offers to keep VIP Champions.
2. **Win-back campaign** – personalised emails with a discount for At Risk customers.
3. **Nurture Potential customers** – 759 customers who could grow into Loyal with follow-up offers.